# Distribution integrity audit
Checks release hashes and parsed row counts. Cleanup definitions and coverage limitations are in README.md. No pattern exploration is performed.

In [ ]:
from pathlib import Path
import csv, json, hashlib
root = Path.cwd()
if root.name != 'distribution': root = root / 'distribution'
manifest = json.loads((root / 'manifest.json').read_text())
for item in manifest:
    path = root / item['file']
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    assert h.hexdigest() == item['sha256']
    with path.open(newline='') as f: count = sum(1 for _ in csv.DictReader(f))
    assert count == item['rows']
    print(item['file'], count, 'verified')
audit = json.loads((root / 'quality_audit.json').read_text())
audit['release_checks']
